# Parameter Sweeps in SymPy + Numeric Simulation

Derive analytically first. Sweep numerically second. Never the reverse.

In [ ]:
from sympy import *
from IPython.display import display, Markdown
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — SymPy Parameter Sweep: lambdify

In [ ]:
section('lambdify: symbolic → fast numeric')

display(Markdown(r'''
`lambdify` converts a SymPy expression to a NumPy function.
Derive symbolically → sweep numerically at C speed.
'''))

# TDGSA convergence vs beta2z
beta2z, n_iter, corr = symbols('beta2z n_iter rho', real=True)

# Empirical model: phase error ~ exp(-|beta2z|/tau) * rho^n
# (illustrative — real behavior from simulation)
tau_conv = 2000  # ps² characteristic scale
phi_err = exp(-Abs(beta2z)/tau_conv) + (1 - exp(-Abs(beta2z)/tau_conv)) * corr**n_iter
step('Phase error model ε(β₂z, n_iter, ρ) =', phi_err)

# Lambdify over beta2z
phi_err_fn = lambdify([beta2z, n_iter, corr], phi_err, 'numpy')

beta2z_vals = np.linspace(-10000, -100, 500)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Sweep 1: beta2z at fixed n_iter=50
for rho in [0.99, 0.95, 0.90, 0.80]:
    axes[0].plot(beta2z_vals, phi_err_fn(beta2z_vals, 50, rho), label=f'ρ={rho}')
axes[0].axvline(-5000, color='k', linestyle='--', label='|β₂z|=5000 ps²')
axes[0].set_xlabel('β₂z (ps²)')
axes[0].set_ylabel('Phase error ε')
axes[0].set_title('Sweep β₂z, n_iter=50')
axes[0].legend(fontsize=8)

# Sweep 2: n_iter at fixed beta2z=-5000
n_vals = np.arange(1, 100)
for rho in [0.99, 0.95, 0.90]:
    axes[1].plot(n_vals, phi_err_fn(-5000, n_vals, rho), label=f'ρ={rho}')
axes[1].set_xlabel('n_iter')
axes[1].set_ylabel('Phase error ε')
axes[1].set_title('Sweep n_iter, |β₂z|=5000 ps²')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('param_sweep_tdgsa.png', dpi=100)
plt.show()
print('lambdify: derived symbolically, swept over 500×100 points instantly')

## §2 — 2D Parameter Sweep: Heatmap

In [ ]:
section('2D sweep: β₂z vs n_iter heatmap')

# Grid
b2z_grid = np.linspace(-10000, -500, 80)
nit_grid = np.arange(1, 81)
B, N = np.meshgrid(b2z_grid, nit_grid)

# Correlation model: rho = exp(-|beta2z|/3000)
rho_grid = np.exp(-np.abs(B)/3000)
err_grid = phi_err_fn(B, N, rho_grid)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.contourf(b2z_grid, nit_grid, err_grid, levels=30, cmap='viridis')
plt.colorbar(im, ax=ax, label='Phase error ε')
ax.axvline(-5000, color='red', linewidth=2, label='β₂z = −5000 ps²')
ax.axhline(50, color='white', linewidth=2, linestyle='--', label='n_iter = 50')
ax.set_xlabel('β₂z (ps²)')
ax.set_ylabel('n_iter')
ax.set_title('TDGSA phase error: 2D parameter space')
ax.legend()
plt.tight_layout()
plt.savefig('param_heatmap.png', dpi=100)
plt.show()
print('Operating point (β₂z=−5000, n_iter=50) marked in red/white')

## §3 — Sensitivity Analysis: ∂ε/∂(param) Symbolically

In [ ]:
section('Symbolic sensitivity: which parameter matters most')

# Which parameter has the largest effect at the operating point?
d_dbeta = diff(phi_err, beta2z)
d_dniter = diff(phi_err, n_iter)
d_drho = diff(phi_err, corr)

step('∂ε/∂(β₂z) =', simplify(d_dbeta))
step('∂ε/∂(n_iter) =', simplify(d_dniter))
step('∂ε/∂ρ =', simplify(d_drho))

# Evaluate sensitivities at operating point
op = {beta2z: -5000, n_iter: 50, corr: 0.85}
s_b2 = float(d_dbeta.subs(op).evalf())
s_ni = float(d_dniter.subs(op).evalf())
s_rh = float(d_drho.subs(op).evalf())

print(f'At operating point (β₂z=−5000, n_iter=50, ρ=0.85):')
print(f'  ∂ε/∂(β₂z)   = {s_b2:.6f}  per ps²')
print(f'  ∂ε/∂(n_iter) = {s_ni:.6f}')
print(f'  ∂ε/∂ρ        = {s_rh:.6f}')
print(f'\nLargest sensitivity: {["β₂z","n_iter","ρ"][np.argmax(np.abs([s_b2,s_ni,s_rh]))]}')

## §4 — SPICE Parameter Sweep via Python

In [ ]:
section('SPICE netlist generation from Python')

display(Markdown(r'''
Generate SPICE netlists programmatically, sweep component values,
parse output — full design-of-experiments from Python.
'''))

def write_rc_netlist(R_val, C_val, filename='rc_sweep.cir'):
    """Write ngspice netlist for RC low-pass, AC sweep."""
    netlist = f"""RC Low-Pass Filter — R={R_val} C={C_val}
.param R={R_val} C={C_val}
Vin in 0 AC 1
R1 in out {{R}}
C1 out 0 {{C}}
.ac dec 100 1 1e9
.print ac vdb(out)
.end
"""
    with open(filename, 'w') as f:
        f.write(netlist)
    return filename

# Sweep R and C, compute cutoff frequency symbolically first
R_sym, C_sym = symbols('R C', positive=True)
f_c = 1 / (2*pi*R_sym*C_sym)
step('RC cutoff frequency: f_c = 1/(2πRC) =', f_c)

f_c_fn = lambdify([R_sym, C_sym], f_c, 'numpy')

R_vals = np.logspace(1, 4, 5)   # 10Ω to 10kΩ
C_vals = np.logspace(-12, -9, 4) # 1pF to 1nF

print('\nRC cutoff frequency sweep (Hz):')
print(f'{"R/C":>10}', end='')
for c in C_vals:
    print(f'  {c:.0e}F', end='')
print()
for r in R_vals:
    print(f'{r:>10.0f}Ω', end='')
    for c in C_vals:
        fc = f_c_fn(r, c)
        if fc >= 1e9:
            print(f'  {fc/1e9:.2f}GHz', end='')
        elif fc >= 1e6:
            print(f'  {fc/1e6:.2f}MHz', end='')
        else:
            print(f'  {fc/1e3:.1f}kHz', end='')
    print()

# Write one netlist
write_rc_netlist('1k', '1n')
print('\nWrote rc_sweep.cir — run with: ngspice rc_sweep.cir')

## §5 — THz Sensor: Parameter Space for TS-DFT

In [ ]:
section('THz TS-DFT design space')

display(Markdown(r'''
Design variables for a time-stretch ADC operating at THz-equivalent rates:
- $f_{rep}$: laser repetition rate
- $T_0$: pulse duration (before stretch)
- $M = |\beta_2 z_2|/|\beta_2 z_1|$: stretch factor
- $f_{ADC}$: ADC sample rate
- $B_{opt}$: optical bandwidth
'''))

f_rep, T0_s, M_s, f_adc, B_opt = symbols('f_rep T_0 M f_ADC B_opt', positive=True)

# Equivalent input bandwidth
B_input = M_s * f_adc / 2
step('Equivalent input bandwidth: B_in = M·f_ADC/2 =', B_input)

# Optical bandwidth constraint
# B_opt = 1/T0 (time-bandwidth product ~ 1 for transform-limited)
B_opt_expr = 1 / T0_s
step('Optical bandwidth (TL pulse): B_opt ≈ 1/T₀ =', B_opt_expr)

# Stretch factor from fiber
beta2_fiber, z1, z2 = symbols('beta_2 z_1 z_2', positive=True)
M_fiber = (beta2_fiber * z2) / (beta2_fiber * z1)
step('Stretch factor M = z₂/z₁ (same fiber) =', M_fiber)

# Numeric sweep: what ADC rate do we need?
print('\nRequired ADC rate for given stretch factor and input bandwidth:')
print(f'{"M":>8} | {"B_in=100GHz":>14} | {"B_in=1THz":>12}')
print('-'*40)
for M_val in [10, 100, 500, 1000, 5000]:
    f_adc_100G = 2*100e9/M_val
    f_adc_1T   = 2*1e12/M_val
    print(f'{M_val:>8} | {f_adc_100G/1e9:>12.1f} GHz | {f_adc_1T/1e9:>10.1f} GHz')

print('\nConclusion: M=1000, 1 THz input → 2 GHz ADC (achievable with 8-bit ADC)')

## §6 — Geometric Proof: ∫₋∞^∞ e^(-x²)dx = √π

In [ ]:
section('Gaussian integral: the geometric proof')

display(Markdown(r'''
**Claim**: $\int_{-\infty}^{\infty} e^{-x^2}\,dx = \sqrt{\pi}$

**Proof** (squaring trick — a geometric argument in 2D):

Let $I = \int_{-\infty}^{\infty} e^{-x^2}dx$. Then:
$$I^2 = \left(\int_{-\infty}^{\infty}e^{-x^2}dx\right)\left(\int_{-\infty}^{\infty}e^{-y^2}dy\right)
= \int_{-\infty}^{\infty}\int_{-\infty}^{\infty}e^{-(x^2+y^2)}\,dx\,dy$$

Switch to polar: $x^2+y^2=r^2$, $dx\,dy = r\,dr\,d\theta$:
$$I^2 = \int_0^{2\pi}d\theta\int_0^{\infty}e^{-r^2}r\,dr$$
'''))

r, theta = symbols('r theta', positive=True)

inner = integrate(r*exp(-r**2), (r, 0, oo))
step('∫₀^∞ r·e^(−r²) dr =', inner)

outer = integrate(inner, (theta, 0, 2*pi))
step('∫₀^(2π) dθ · (1/2) = I² =', outer)

I_val = sqrt(outer)
step('I = √π =', I_val)

# Verify directly
x = symbols('x', real=True)
step('SymPy direct: ∫₋∞^∞ e^(−x²) dx =', integrate(exp(-x**2), (x, -oo, oo)))

display(Markdown(r'''
**Why this is geometric**: $I^2$ is the volume under $e^{-(x^2+y^2)}$ over the entire plane.
The integrand is circularly symmetric → polar coordinates factor the integral.
The $r\,dr$ Jacobian (area element in polar) is what makes it solvable.

This integral appears everywhere in these notebooks:
- Gaussian PDF normalization
- Quantum harmonic oscillator wavefunctions  
- Fraunhofer diffraction from a Gaussian aperture
- Thermal noise power (equipartition)
- Cramér-Rao bound for Gaussian measurements
'''))